# P1 feedback experiments: additive server runs

The paper now includes the completed Q16 public-clock planning diagnostic, which misses its observed timely-service objective. This notebook prepares the remaining fixed-phase baseline and application replication. **No new native result has been claimed.**

Extract the companion ZIP, then open this notebook from its package directory in the existing NetSquid/Qoala kernel. Existing Q12/Q13/Q16 files are never modified. The default is a smoke run; full runs require the explicit mode below and a matching successful smoke receipt.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, signal

PACKAGE = Path.cwd().resolve()
if not (PACKAGE / "source/closure_runner.py").exists():
    candidates = list(PACKAGE.glob("*/source/closure_runner.py"))
    if len(candidates) != 1:
        raise RuntimeError("Open this notebook inside the extracted P1_Feedback_Closure package")
    PACKAGE = candidates[0].parents[1]

FAMILY = "timing"        # "timing" or "application"
MODE = "smoke"           # Change explicitly to "full" only after reading the frozen design
WORKERS = 8
QOALA_REPO = None         # Set a path only if automatic repository discovery fails
RUN_ROOT = PACKAGE / "runs"
RUN_DIR = RUN_ROOT / (FAMILY + "-" + MODE)
SMOKE_DIR = RUN_ROOT / (FAMILY + "-smoke")
print({"package": str(PACKAGE), "python": sys.executable, "run": str(RUN_DIR)})


## Check the package and native environment

Offline tests check the information contract, pairing, statistical formula, missingness and checkpoint behavior. They do not validate NetSquid. The native preflight requires the pinned clean Qoala source and the imported native dependencies. There is no fallback that generates synthetic experimental outcomes.


In [ ]:
subprocess.run([sys.executable, str(PACKAGE / "source/offline_tests.py")], cwd=PACKAGE, check=True)
preflight = [sys.executable, str(PACKAGE / "source/closure_runner.py"), "preflight",
             "--folder", str(RUN_ROOT / "environment")]
if QOALA_REPO:
    preflight += ["--repo", str(QOALA_REPO)]
subprocess.run(preflight, cwd=PACKAGE, check=True)


## Fixed design and compute scale

Timing uses 16,384 fresh paired trials per workload and 20 arms: learned, 16 fixed phases, random, even and idle. The 36 simultaneous comparisons condition on the frozen ensemble of old policies. Application replication uses 8,192 fresh whole streams, four configurations, five policies and four arms; its six primary comparisons are frozen in the source. Each full study contains **655,360 native executions**. These can require substantial compute and storage. Inspect smoke time and output size before choosing full mode.

Application N targets a 2 percentage point half-width under a fourfold historical variance envelope; it is not a power guarantee. The current Q16 planning data remain excluded from confirmation. No sample-size extension or retuning based on future evaluation results is permitted.


In [ ]:
precision = json.loads((PACKAGE / "evidence/precision_plan.json").read_text())
print(json.dumps(precision, indent=2))
if SMOKE_DIR.exists() and (SMOKE_DIR / "COMPLETE.json").exists():
    receipts = [json.loads(p.read_text()) for p in (SMOKE_DIR / "checkpoints").glob("*.json")]
    seconds = sum(r["wall_seconds"] for r in receipts)
    executions = sum(r["native_executions"] for r in receipts)
    total = precision["native_executions_full"][FAMILY]
    raw_bytes = sum(p.stat().st_size for p in (SMOKE_DIR / "raw").glob("*.gz"))
    print({"rough_serial_hours": seconds / executions * total / 3600,
           "ideal_parallel_hours": seconds / executions * total / (3600 * WORKERS),
           "rough_raw_GB": raw_bytes / executions * total / 1e9,
           "caution": "Smoke extrapolation only; neither runtime nor storage is guaranteed"})


## Run the selected stage

This cell executes only the FAMILY and MODE selected above. Repeating it resumes verified completed checkpoints. Source or protocol changes require a new folder and another smoke gate. Engineering failures preserve partial raw records and stop the run. Smoke outputs have no effect inference.


In [ ]:
if FAMILY not in ("timing", "application") or MODE not in ("smoke", "full"):
    raise ValueError("Unknown study or mode")
command = [sys.executable, str(PACKAGE / "source/closure_runner.py"), "run",
           "--family", FAMILY, "--mode", MODE, "--workers", str(WORKERS), "--folder", str(RUN_DIR)]
if MODE == "full":
    if not (SMOKE_DIR / "COMPLETE.json").exists():
        raise RuntimeError("Run the matching smoke stage first")
    command += ["--smoke", str(SMOKE_DIR)]
if QOALA_REPO:
    command += ["--repo", str(QOALA_REPO)]
RUN_DIR.mkdir(parents=True, exist_ok=True)
with (RUN_DIR / "console.log").open("a") as log:
    process = subprocess.Popen(command, cwd=PACKAGE, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="")
            log.write(line); log.flush()
        if process.wait() != 0:
            raise RuntimeError("Native stage failed; inspect console.log and failures/")
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGTERM)
        raise


## Inspect and export compact review evidence

An effect claim requires a complete fixed-N run and an independent audit of its native records. A smoke PASS establishes integration only. Missing endpoints remain in all-offered accounting and block the affected effect claim. The review ZIP omits large native raw traces; keep raw/ on the server for the next audit.


In [ ]:
import zipfile
analysis = json.loads((RUN_DIR / "ANALYSIS.json").read_text())
print(json.dumps(analysis, indent=2))
archive = RUN_DIR.with_name(RUN_DIR.name + "-compact-review.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob("*"):
        if p.is_file() and "raw" not in p.relative_to(RUN_DIR).parts and not p.name.endswith(".tmp"):
            z.write(p, p.relative_to(RUN_DIR))
print(archive)
